In [32]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error


In [35]:
df = pd.read_csv("cleaned_crmls_sold.csv")

train_df = df[df['is_test'] == 0].copy()
test_df = df[df['is_test'] == 1].copy()

feature_cols = [col for col in df.columns if col not in ['target_log_price', 'is_test']]

X_train = train_df[feature_cols]
y_train = train_df['target_log_price']

X_test = test_df[feature_cols]
y_test = test_df['target_log_price']

y_test_original = np.expm1(y_test)
df.head()

,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeAcres,PropertyType_Residential,PropertySubType_SingleFamilyResidence,target_log_price,is_test
0,-0.093094,0.526197,-0.568203,-0.017498,1.0,1.0,12.821261,0
1,-0.538801,-1.546351,-0.568203,-0.018761,1.0,1.0,13.664689,0
2,-0.804492,0.526197,-0.568203,-0.018789,1.0,1.0,13.102163,0
3,1.488539,2.598745,2.073429,-0.018753,1.0,1.0,13.345509,0
4,0.036864,-0.510077,0.312341,-0.018959,1.0,1.0,14.051279,0


In [36]:
models = {
    "Baseline (Linear Regression)": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42, max_depth=10),
    "Random Forest Regressor": RandomForestRegressor(random_state=42, n_estimators=100, max_depth=15, n_jobs=-1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred_log = model.predict(X_test)
    r2 = r2_score(y_test, y_pred_log)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_log))
    
    y_pred_original = np.expm1(y_pred_log)
    mae_dollar = mean_absolute_error(y_test_original, y_pred_original)
    medae_dollar = median_absolute_error(y_test_original, y_pred_original)
    
    results.append({
        "Model": name,
        "Test R² Score": round(r2, 4),
        "RMSE (Log)": round(rmse, 4),
        "MAE ($)": f"${mae_dollar:,.2f}",
        "Median AE ($)": f"${medae_dollar:,.2f}"
    })

results_df = pd.DataFrame(results)
print("=" * 70)
print("📊 Week 5 Mpdel result (vs Baseline)")
print("=" * 70)
print(results_df.to_string(index=False))

📊 Week 5 Mpdel result (vs Baseline)
                       Model  Test R² Score  RMSE (Log)     MAE ($) Median AE ($)
Baseline (Linear Regression)         0.3535      0.5499 $906,764.03   $343,557.35
     Decision Tree Regressor         0.3488      0.5519 $638,530.70   $327,563.68
     Random Forest Regressor         0.4068      0.5268 $535,075.56   $306,759.44


In [37]:
rf_model = models["Random Forest Regressor"]
importances = rf_model.feature_importances_

feature_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_imp.head(10).to_string(index=False))

                              Feature  Importance
                           LivingArea    0.410533
                BathroomsTotalInteger    0.314796
                         LotSizeAcres    0.235436
                        BedroomsTotal    0.039235
             PropertyType_Residential    0.000000
PropertySubType_SingleFamilyResidence    0.000000


In [38]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error

In [44]:
df_fe = df.copy()

if 'LivingArea' in df_fe.columns and 'Bedrooms' in df_fe.columns:
    df_fe['living_area_x_bedrooms'] = df_fe['LivingArea'] * df_fe['Bedrooms']

if 'LivingArea' in df_fe.columns and 'TotalBathroomsTotalInteger' in df_fe.columns:
    df_fe['living_area_x_bathrooms'] = df_fe['LivingArea'] * df_fe['TotalBathroomsTotalInteger']

if 'Bedrooms' in df_fe.columns and 'TotalBathroomsTotalInteger' in df_fe.columns:
    df_fe['total_rooms_index'] = df_fe['Bedrooms'] + df_fe['TotalBathroomsTotalInteger']

if 'LivingArea' in df_fe.columns and 'LotSizeAcres' in df_fe.columns:
    df_fe['living_to_lot_ratio'] = df_fe['LivingArea'] - df_fe['LotSizeAcres']

In [45]:
train_df_new = df_fe[df_fe['is_test'] == 0].copy()
test_df_new = df_fe[df_fe['is_test'] == 1].copy()

feature_cols_new = [col for col in df_fe.columns if col not in ['target_log_price', 'is_test']]

X_train_new = train_df_new[feature_cols_new]
y_train_new = train_df_new['target_log_price']

X_test_new = test_df_new[feature_cols_new]
y_test_new = test_df_new['target_log_price']
y_test_orig_new = np.expm1(y_test_new)

models_new = {
    "Baseline (Linear Regression)": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42, max_depth=10),
    "Random Forest Regressor": RandomForestRegressor(random_state=42, n_estimators=100, max_depth=15, n_jobs=-1)
}

comparison_results = []

for i, (name, model) in enumerate(models_new.items()):
    model.fit(X_train_new, y_train_new)
    
    y_pred_log_new = model.predict(X_test_new)
    r2_new = r2_score(y_test_new, y_pred_log_new)
    
    y_pred_orig_new = np.expm1(y_pred_log_new)
    mae_new = mean_absolute_error(y_test_orig_new, y_pred_orig_new)
    medae_new = median_absolute_error(y_test_orig_new, y_pred_orig_new)

    old_r2 = results[i]["Test R² Score"]
    old_medae = results[i]["Median AE ($)"]
    
    comparison_results.append({
        "Model": name,
        "Old R²": old_r2,
        "New R²": round(r2_new, 4),
        "R² Change": f"{(r2_new - old_r2):+.4f}",
        "Old Median AE": old_medae,
        "New Median AE": f"${medae_new:,.2f}"
    })

In [46]:
comp_df = pd.DataFrame(comparison_results)

print(comp_df.to_string(index=False))

rf_model_new = models_new["Random Forest Regressor"]
feature_imp_new = pd.DataFrame({
    'Feature': feature_cols_new,
    'Importance': rf_model_new.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n" + "=" * 80)
print("Feature Importance")
print("=" * 80)
print(feature_imp_new.head(10).to_string(index=False))

                       Model  Old R²  New R² R² Change Old Median AE New Median AE
Baseline (Linear Regression)  0.3535  0.3536   +0.0001   $343,557.35   $343,833.63
     Decision Tree Regressor  0.3488  0.3524   +0.0036   $327,563.68   $326,771.99
     Random Forest Regressor  0.4068  0.4074   +0.0006   $306,759.44   $308,046.11

Feature Importance
                              Feature  Importance
                BathroomsTotalInteger    0.315284
                  living_to_lot_ratio    0.274186
                         LotSizeAcres    0.216247
                           LivingArea    0.157550
                        BedroomsTotal    0.036733
             PropertyType_Residential    0.000000
PropertySubType_SingleFamilyResidence    0.000000
